# ParkShare ABM Simulation — OSM Street Network Validation

**Author:** Ana  
**Project:** ParkShare — Peer-to-Peer Real-Time Parking Exchange (TFG, IE University)  
**Description:** Robustness check replacing the synthetic 20×20 grid with the real street network of **Barrio de Salamanca, Madrid**, downloaded from OpenStreetMap via OSMnx.

Key differences from v1/v2:
- Agents move along actual street edges (weighted by travel time)
- Match radius is expressed in metres (300 m) rather than grid units
- Step-to-minute conversion is **auto-calibrated** to the Shoup (2006) empirical baseline of 8.6 min mean search time
- All other parameters (30 Monte Carlo runs, 94% initial occupancy, Poisson arrivals) are held constant for comparability

Results from this notebook are reported in the **Robustness & Limitations** section of the thesis.

---

## 1. Imports & Global Settings

In [ ]:
# ============================================================
#  ParkShare Madrid — OSM Street Network Simulation
#  Full simulation on real Madrid street geometry (Salamanca)
#  with automatic calibration to Shoup (2006) 8.6-min baseline
# ============================================================

import subprocess
subprocess.run(["pip", "install", "osmnx", "-q"])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import random
import networkx as nx
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

SEED = 42

PS_GRAY  = '#c8d8e4'
PS_LIGHT = '#b0cfe8'
PS_MID   = '#5ba3d0'
PS_DARK  = '#2a6496'
BAR_C    = [PS_GRAY, PS_LIGHT, PS_LIGHT, PS_MID, PS_DARK]

mpl.rcParams.update({
    'font.family':'sans-serif','font.size':10,
    'axes.titlesize':11,'axes.titleweight':'bold','axes.titlepad':12,
    'axes.labelsize':9,'axes.labelcolor':'#666',
    'xtick.labelsize':9,'ytick.labelsize':9,
    'xtick.color':'#888','ytick.color':'#888',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.spines.left':False,'axes.spines.bottom':False,
    'axes.grid':True,'axes.grid.axis':'y',
    'grid.color':'#e8e8e8','grid.linewidth':0.6,
    'xtick.bottom':False,'ytick.left':False,
    'figure.dpi':300,'figure.facecolor':'white',
})

SRC  = 'ParkShare OSM simulation  ·  Salamanca, Madrid  ·  30 Monte Carlo runs'
KW_S = dict(fontsize=7.5, color='#999', ha='left')
KW_L = dict(ha='center', va='bottom', fontsize=9, fontweight='bold', color='#333')

# ============================================================
#  SIMULATION PARAMETERS
# ============================================================

N_SPOTS   = 55
INIT_OCC  = 0.94
N_RUNS    = 30
N_STEPS   = 600
ARR_RATE  = 0.10
DEP_PROB  = 0.006
RADIUS_M  = 300    # match radius in metres (real world)
COST_PRIV = 3.0
COST_P2P  = 3.0
CO2_KM    = 0.12
MAX_MIN   = 40.0

TARGET_BASELINE = 8.6   # Shoup (2006) empirical anchor

RATES  = [0.0, 0.10, 0.30, 0.60, 1.0]
LABELS = ['0%', '10%', '30%', '60%', '100%']

# ============================================================
#  STEP 1 — DOWNLOAD OSM STREET NETWORK
# ============================================================

print("=" * 60)
print("STEP 1: Downloading OSM street network")
print("=" * 60)

import osmnx as ox

G = ox.graph_from_bbox(
    bbox=(40.4380, 40.4180, -3.6700, -3.6950),
    network_type='drive'
)
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)
nodes, edges = ox.graph_to_gdfs(G)

print(f"  Network: {len(G.nodes)} nodes, {len(G.edges)} edges")
print(f"  Bounding box: "
      f"lon [{nodes.x.min():.4f}, {nodes.x.max():.4f}]  "
      f"lat [{nodes.y.min():.4f}, {nodes.y.max():.4f}]")

# Place parking spots on real street nodes
rng_init   = random.Random(SEED)
all_nodes  = list(G.nodes(data=True))
spot_sample = rng_init.sample(all_nodes, N_SPOTS)
spot_ids    = [n    for n, _ in spot_sample]
spot_coords = [(d['x'], d['y']) for _, d in spot_sample]

print(f"  Placed {N_SPOTS} parking spots on real street nodes")

# Pre-compute shortest path lengths between all spot nodes and all nodes
# (expensive but done once — cached for all runs)
print("  Pre-computing shortest path distances (this takes ~1 min)...")
all_node_ids = [n for n, _ in all_nodes]

# For each spot, get distances FROM that spot to all reachable nodes
spot_distances = {}
for sid in spot_ids:
    try:
        lengths = nx.single_source_dijkstra_path_length(
            G, sid, cutoff=RADIUS_M * 1.5, weight='length')
        spot_distances[sid] = lengths
    except Exception:
        spot_distances[sid] = {}

print(f"  Distance cache built for {len(spot_distances)} spots")


# ============================================================
#  CORE OSM SIMULATION FUNCTION
# ============================================================

def run_osm_episode(rate, seed, step_to_min):
    rng    = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    spots = [{'node': nid, 'x': x, 'y': y,
              'occ': rng.random() < INIT_OCC, 'res': None}
             for nid, (x, y) in zip(spot_ids, spot_coords)]

    drivers, completed, nid = [], [], 0

    for _ in range(N_STEPS):

        # 1. Poisson arrivals — random street node
        for _ in range(np_rng.poisson(ARR_RATE)):
            start_node, start_data = rng.choice(all_nodes)
            drivers.append({
                'id': nid, 'app': rng.random() < rate,
                'node': start_node,
                'x': start_data['x'], 'y': start_data['y'],
                'state': 'searching',
                'ss': 0, 'path_len_m': 0.0,
                'target': None
            })
            nid += 1

        # 2. Spontaneous departures
        for s in spots:
            if s['occ'] and s['res'] is None and rng.random() < DEP_PROB:
                s['occ'] = False

        # 3. Agent transitions
        done = []
        for d in drivers:
            if d['state'] in ('parked', 'gave_up'):
                done.append(d); continue

            d['ss'] += 1
            sm = d['ss'] * step_to_min

            if sm > MAX_MIN:
                d['state'] = 'gave_up'
                if d['target']:
                    d['target']['res'] = None
                    d['target'] = None
                done.append(d); continue

            # App agents: find nearest free spot within RADIUS_M metres
            if d['app'] and d['state'] == 'searching':
                free = [s for s in spots
                        if not s['occ'] and s['res'] is None]
                if free:
                    best_s, best_dist = None, float('inf')
                    try:
                        lengths_from_driver = nx.single_source_dijkstra_path_length(
                            G, d['node'], cutoff=RADIUS_M, weight='length')
                    except Exception:
                        lengths_from_driver = {}

                    for s in free:
                        dist = lengths_from_driver.get(s['node'], float('inf'))
                        if dist < best_dist:
                            best_dist = dist
                            best_s    = s

                    if best_s and best_dist <= RADIUS_M:
                        best_s['res']     = d['id']
                        d['target']       = best_s
                        d['path_len_m']  += best_dist
                        d['state']        = 'matched'

            # Matched: move directly to spot (next step = parked)
            if d['state'] == 'matched' and d['target']:
                d['target']['occ'] = True
                d['target']['res'] = None
                d['state']  = 'parked'
                d['target'] = None
                done.append(d)
                continue

            # No-app agents: random walk on street network
            if d['state'] == 'searching' and not d['app']:
                neighbors = list(G.successors(d['node']))
                if neighbors:
                    next_node = rng.choice(neighbors)
                    edge_data = G.get_edge_data(d['node'], next_node, 0)
                    edge_len  = edge_data.get('length', 50) if edge_data else 50
                    d['path_len_m'] += edge_len
                    d['node'] = next_node
                    nd = G.nodes[next_node]
                    d['x'], d['y'] = nd['x'], nd['y']

                # Park if current node is a free spot
                adj = [s for s in spots
                       if not s['occ'] and s['res'] is None
                       and s['node'] == d['node']]
                if adj:
                    adj[0]['occ'] = True
                    d['state'] = 'parked'
                    done.append(d)
                    continue

        # 4. Record completed
        for d in done:
            if d['ss'] > 0:
                sm  = d['ss'] * step_to_min
                dkm = d['path_len_m'] / 1000.0
                gu  = d['state'] == 'gave_up'
                cost = (COST_PRIV * (sm/60 + 0.5)) if gu \
                       else (COST_P2P if d['app'] else 0.5)
                completed.append({
                    'app': d['app'], 'sm': sm, 'dkm': dkm,
                    'cost': cost, 'co2': dkm * CO2_KM, 'gu': gu
                })

        drivers = [d for d in drivers
                   if d['state'] not in ('parked', 'gave_up')]

    return completed


def run_baseline(step_to_min, n_runs=10):
    """Run baseline (0% adoption) and return mean search time."""
    pool = []
    for run in range(n_runs):
        pool.extend(run_osm_episode(0.0, SEED + run * 13, step_to_min))
    return np.mean([r['sm'] for r in pool])


# ============================================================
#  STEP 2 — AUTO-CALIBRATION (binary search on S2M)
# ============================================================

print("\n" + "=" * 60)
print("STEP 2: Auto-calibrating S2M to Shoup (2006) 8.6-min baseline")
print("=" * 60)

s2m_lo, s2m_hi = 0.01, 5.0
s2m_best = 0.60
tolerance = 0.15   # acceptable deviation from 8.6 min

for iteration in range(12):
    s2m_mid  = (s2m_lo + s2m_hi) / 2
    baseline = run_baseline(s2m_mid, n_runs=10)
    print(f"  Iteration {iteration+1:2d}  S2M={s2m_mid:.4f}  "
          f"baseline={baseline:.2f} min  target={TARGET_BASELINE}")

    if abs(baseline - TARGET_BASELINE) < tolerance:
        s2m_best = s2m_mid
        print(f"\n  ✓ Converged: S2M = {s2m_best:.4f}  "
              f"(baseline = {baseline:.2f} min)")
        break

    if baseline > TARGET_BASELINE:
        s2m_hi = s2m_mid
    else:
        s2m_lo = s2m_mid
    s2m_best = s2m_mid

S2M_CALIBRATED = s2m_best
S2KM = S2M_CALIBRATED * (50 / 1000)  # approximate: 1 step ≈ 50m street distance

print(f"\n  Final calibrated S2M = {S2M_CALIBRATED:.4f} min/step")
print(f"  Equivalent S2KM     = {S2KM:.5f} km/step")
print(f"  (Baseline on OSM network = {run_baseline(S2M_CALIBRATED, 10):.2f} min)")


# ============================================================
#  STEP 3 — FULL MONTE CARLO ON OSM NETWORK
# ============================================================

print("\n" + "=" * 60)
print("STEP 3: Full Monte Carlo simulation on OSM street network")
print("=" * 60)

osm_results     = {}
osm_run_results = {}

for rate in RATES:
    pool      = []
    run_means = []
    for run in range(N_RUNS):
        ep = run_osm_episode(rate, SEED + run * 13, S2M_CALIBRATED)
        pool.extend(ep)
        run_means.append(np.mean([r['sm'] for r in ep]))
    osm_results[rate]     = pool
    osm_run_results[rate] = run_means
    t = np.mean([r['sm']  for r in pool])
    d = np.mean([r['dkm'] for r in pool])
    g = 100 * np.mean([r['gu'] for r in pool])
    print(f"  {int(rate*100):3d}%  |  n={len(pool):4d}  |  "
          f"search={t:.2f} min  dist={d:.3f} km  give-up={g:.1f}%")

osm_summary = []
for rate, label in zip(RATES, LABELS):
    d  = osm_results[rate]
    rm = osm_run_results[rate]
    se = stats.sem(rm)
    ci = stats.t.interval(0.95, df=N_RUNS-1, loc=np.mean(rm), scale=se)
    osm_summary.append({
        'Adoption rate':         label,
        'Avg search time (min)': round(np.mean([r['sm']  for r in d]), 2),
        'Avg distance (km)':     round(np.mean([r['dkm'] for r in d]), 3),
        'Avg cost (€)':          round(np.mean([r['cost']for r in d]), 2),
        'Avg CO₂ (kg)':          round(np.mean([r['co2'] for r in d]), 4),
        'Give-up rate (%)':      round(100*np.mean([r['gu'] for r in d]), 1),
        '95% CI lower':          round(ci[0], 2),
        '95% CI upper':          round(ci[1], 2),
    })

df_osm = pd.DataFrame(osm_summary)
print("\n── OSM Results table ──────────────────────────────────")
print(df_osm.to_string(index=False))
df_osm.to_csv('parkshare_osm_full_results.csv', index=False)


# ============================================================
#  STEP 4 — STATISTICAL TESTS
# ============================================================

print("\n" + "=" * 60)
print("STEP 4: Statistical tests")
print("=" * 60)

baseline_runs = osm_run_results[0.0]
stat_rows = []
for rate, label in zip(RATES[1:], LABELS[1:]):
    t_stat, p_val = stats.ttest_ind(baseline_runs, osm_run_results[rate])
    cohens_d = (np.mean(baseline_runs) - np.mean(osm_run_results[rate])) / \
               np.sqrt((np.std(baseline_runs)**2 +
                        np.std(osm_run_results[rate])**2) / 2)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 \
          else '*' if p_val < 0.05 else 'ns'
    print(f"  0% vs {label:4s}  t={t_stat:6.3f}  p={p_val:.6f}  "
          f"d={cohens_d:.2f}  {sig}")
    stat_rows.append({
        'Comparison': f'0% vs {label}',
        't-statistic': round(t_stat, 3),
        'p-value': round(p_val, 6),
        "Cohen's d": round(cohens_d, 2),
        'Significance': sig,
    })

pd.DataFrame(stat_rows).to_csv('parkshare_osm_stats.csv', index=False)


# ============================================================
#  STEP 5 — SENSITIVITY ANALYSIS ON OSM NETWORK
# ============================================================

print("\n" + "=" * 60)
print("STEP 5: Sensitivity analysis on OSM network (60% adoption)")
print("=" * 60)

# For sensitivity on OSM we vary parameters that don't require
# rebuilding the network: N_SPOTS, INIT_OCC, ARR_RATE, DEP_PROB
# RADIUS_M is also varied (changes matching threshold in metres)

def run_osm_sensitivity(rate, seed, step_to_min,
                         n_spots_=N_SPOTS, init_occ_=INIT_OCC,
                         arr_rate_=ARR_RATE, dep_prob_=DEP_PROB,
                         radius_m_=RADIUS_M):
    """Sensitivity variant — overrides key parameters."""
    rng    = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    spot_sample_ = rng.sample(all_nodes, n_spots_)
    spots = [{'node': n, 'x': d['x'], 'y': d['y'],
              'occ': rng.random() < init_occ_, 'res': None}
             for n, d in spot_sample_]

    drivers, completed, nid = [], [], 0

    for _ in range(N_STEPS):
        for _ in range(np_rng.poisson(arr_rate_)):
            sn, sd = rng.choice(all_nodes)
            drivers.append({
                'id': nid, 'app': rng.random() < rate,
                'node': sn, 'x': sd['x'], 'y': sd['y'],
                'state': 'searching', 'ss': 0, 'path_len_m': 0.0,
                'target': None
            })
            nid += 1

        for s in spots:
            if s['occ'] and s['res'] is None and rng.random() < dep_prob_:
                s['occ'] = False

        done = []
        for d in drivers:
            if d['state'] in ('parked', 'gave_up'):
                done.append(d); continue
            d['ss'] += 1
            sm = d['ss'] * step_to_min
            if sm > MAX_MIN:
                d['state'] = 'gave_up'
                if d['target']:
                    d['target']['res'] = None; d['target'] = None
                done.append(d); continue

            if d['app'] and d['state'] == 'searching':
                free = [s for s in spots
                        if not s['occ'] and s['res'] is None]
                if free:
                    best_s, best_dist = None, float('inf')
                    try:
                        lens = nx.single_source_dijkstra_path_length(
                            G, d['node'], cutoff=radius_m_, weight='length')
                    except Exception:
                        lens = {}
                    for s in free:
                        dist = lens.get(s['node'], float('inf'))
                        if dist < best_dist:
                            best_dist = dist; best_s = s
                    if best_s and best_dist <= radius_m_:
                        best_s['res']    = d['id']
                        d['target']      = best_s
                        d['path_len_m'] += best_dist
                        d['state']       = 'matched'

            if d['state'] == 'matched' and d['target']:
                d['target']['occ'] = True
                d['target']['res'] = None
                d['state']  = 'parked'
                d['target'] = None
                done.append(d); continue

            if d['state'] == 'searching' and not d['app']:
                nbrs = list(G.successors(d['node']))
                if nbrs:
                    nxt = rng.choice(nbrs)
                    ed  = G.get_edge_data(d['node'], nxt, 0)
                    d['path_len_m'] += ed.get('length', 50) if ed else 50
                    d['node'] = nxt
                    nd = G.nodes[nxt]
                    d['x'], d['y'] = nd['x'], nd['y']
                adj = [s for s in spots
                       if not s['occ'] and s['res'] is None
                       and s['node'] == d['node']]
                if adj:
                    adj[0]['occ'] = True
                    d['state'] = 'parked'
                    done.append(d); continue

        for d in done:
            if d['ss'] > 0:
                sm  = d['ss'] * step_to_min
                dkm = d['path_len_m'] / 1000.0
                gu  = d['state'] == 'gave_up'
                cost = (COST_PRIV * (sm/60 + 0.5)) if gu \
                       else (COST_P2P if d['app'] else 0.5)
                completed.append({'app': d['app'], 'sm': sm, 'dkm': dkm,
                                  'cost': cost, 'co2': dkm*CO2_KM, 'gu': gu})
        drivers = [d for d in drivers
                   if d['state'] not in ('parked', 'gave_up')]
    return completed

sensitivity_params = {
    'N_SPOTS\n(parking supply)': {
        'kwarg':  'n_spots_',
        'values': [40, 55, 70],
        'labels': ['Low\n(40)', 'Baseline\n(55)', 'High\n(70)'],
    },
    'INIT_OCC\n(peak occupancy)': {
        'kwarg':  'init_occ_',
        'values': [0.85, 0.94, 0.99],
        'labels': ['Low\n(85%)', 'Base\n(94%)', 'High\n(99%)'],
    },
    'RADIUS\n(match radius)': {
        'kwarg':  'radius_m_',
        'values': [150, 300, 500],
        'labels': ['150m', '300m\n(base)', '500m'],
    },
    'ARR_RATE\n(demand)': {
        'kwarg':  'arr_rate_',
        'values': [0.07, 0.10, 0.13],
        'labels': ['Low\n(0.07)', 'Base\n(0.10)', 'High\n(0.13)'],
    },
    'DEP_PROB\n(turnover)': {
        'kwarg':  'dep_prob_',
        'values': [0.003, 0.006, 0.010],
        'labels': ['Low\n(0.003)', 'Base\n(0.006)', 'High\n(0.010)'],
    },
}

sens_results = {}
for param_name, cfg in sensitivity_params.items():
    row_means, row_ci_lo, row_ci_hi = [], [], []
    for v in cfg['values']:
        run_ms = []
        for run in range(N_RUNS):
            ep = run_osm_sensitivity(0.60, SEED + run * 13,
                                      S2M_CALIBRATED,
                                      **{cfg['kwarg']: v})
            run_ms.append(np.mean([r['sm'] for r in ep]))
        m  = np.mean(run_ms)
        se = stats.sem(run_ms)
        ci = stats.t.interval(0.95, df=N_RUNS-1, loc=m, scale=se)
        row_means.append(m)
        row_ci_lo.append(ci[0])
        row_ci_hi.append(ci[1])
        print(f"  {param_name.split(chr(10))[0]:12s} = {v}  →  "
              f"{m:.2f} min  CI=[{ci[0]:.2f}, {ci[1]:.2f}]")
    sens_results[param_name] = {
        'means': row_means, 'ci_lo': row_ci_lo,
        'ci_hi': row_ci_hi, 'labels': cfg['labels'],
    }


# ============================================================
#  CHARTS
# ============================================================

print("\n── Saving charts ───────────────────────────────────────")

times = [s['Avg search time (min)'] for s in osm_summary]
dists = [s['Avg distance (km)']     for s in osm_summary]
costs = [s['Avg cost (€)']          for s in osm_summary]
co2s  = [s['Avg CO₂ (kg)']          for s in osm_summary]
ci_lo = [s['95% CI lower']          for s in osm_summary]
ci_hi = [s['95% CI upper']          for s in osm_summary]

def save_chart(fname, vals, ylabel, title, source, fmt,
               ci_lo=None, ci_hi=None, extra=''):
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    bars = ax.bar(LABELS, vals, color=BAR_C, width=0.5,
                  zorder=3, edgecolor='white')
    if ci_lo and ci_hi:
        ax.errorbar(LABELS, vals,
                    yerr=[[v-l for v,l in zip(vals,ci_lo)],
                           [h-v for v,h in zip(vals,ci_hi)]],
                    fmt='none', color='#444', capsize=4,
                    linewidth=1.2, zorder=4)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2,
                b.get_height()+max(vals)*0.025,
                fmt.format(v), **KW_L)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('ParkShare adoption rate', labelpad=8)
    ax.set_ylim(0, max(vals)*1.22)
    ax.set_title(title, loc='center')
    fig.text(0.02, -0.02, source + extra, **KW_S)
    fig.tight_layout()
    fig.savefig(fname, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"  Saved: {fname}")

save_chart('osm_A_search_time.png', times, 'Minutes',
           'Average parking search time by adoption rate\n'
           f'(OSM network · Salamanca, Madrid · S2M={S2M_CALIBRATED:.4f})',
           SRC, '{:.2f} min', ci_lo=ci_lo, ci_hi=ci_hi)

save_chart('osm_B_distance.png', dists, 'Kilometres',
           'Average cruising distance by adoption rate\n(real street distances, OSM)',
           SRC, '{:.3f} km')

save_chart('osm_C_cost.png', costs, 'Euros (€)',
           'Average parking cost per driver\n(OSM network)',
           SRC, '€{:.2f}',
           extra='  ·  Private = €3/hr  ·  P2P = €3 flat')

save_chart('osm_D_co2.png', co2s, 'kg CO₂',
           'Average CO₂ emissions per driver\n(real street distances)',
           SRC, '{:.4f} kg',
           extra='  ·  0.12 kg CO₂/km (EEA 2023)')

# Statistical significance chart
fig, ax = plt.subplots(figsize=(6.5, 3.6))
comparisons = [r['Comparison']  for r in stat_rows]
pvals       = [r['p-value']     for r in stat_rows]
cohens      = [r["Cohen's d"]   for r in stat_rows]
bar_cols    = [PS_DARK if p < 0.001 else PS_MID if p < 0.01
               else PS_LIGHT for p in pvals]
bars = ax.bar(comparisons, [-np.log10(max(p, 1e-10)) for p in pvals],
              color=bar_cols, width=0.5, zorder=3, edgecolor='white')
ax.axhline(y=-np.log10(0.05),  color='#e07b39', linewidth=1,
           linestyle='--', zorder=4, label='p = 0.05')
ax.axhline(y=-np.log10(0.001), color='#c0392b', linewidth=1,
           linestyle=':', zorder=4, label='p = 0.001')
for b, p, d in zip(bars, pvals, cohens):
    ax.text(b.get_x()+b.get_width()/2,
            b.get_height()+0.15, f'd={d:.2f}', **KW_L)
ax.set_ylabel('−log₁₀(p-value)')
ax.set_xlabel('Comparison vs baseline (0% adoption)', labelpad=8)
ax.set_title('Statistical significance of adoption effect\n'
             '(two-sample t-test, 30 runs · OSM network)', loc='center')
ax.legend(fontsize=8, frameon=False)
fig.text(0.02, -0.02, SRC + "  ·  Cohen's d above bars", **KW_S)
fig.tight_layout()
fig.savefig('osm_E_significance.png', bbox_inches='tight', dpi=300)
plt.close()
print("  Saved: osm_E_significance.png")

# Sensitivity analysis chart
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
fig.suptitle(
    'Sensitivity analysis — mean search time at 60% adoption\n'
    'OSM street network · Salamanca, Madrid · 95% CI (30 runs)',
    fontsize=11, fontweight='bold', y=1.02
)
for ax, (param_name, sr) in zip(axes, sens_results.items()):
    short = param_name.split('\n')[0]
    cols  = [PS_LIGHT, PS_DARK, PS_MID]
    bars  = ax.bar(sr['labels'], sr['means'], color=cols,
                   width=0.5, zorder=3, edgecolor='white')
    ax.errorbar(sr['labels'], sr['means'],
                yerr=[[m-l for m,l in zip(sr['means'],sr['ci_lo'])],
                       [h-m for m,h in zip(sr['means'],sr['ci_hi'])]],
                fmt='none', color='#444', capsize=4, linewidth=1.2, zorder=4)
    for b, v in zip(bars, sr['means']):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
                f'{v:.1f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold', color='#333')
    ax.set_title(short, fontsize=9, fontweight='bold')
    ax.set_ylabel('Min' if ax == axes[0] else '', fontsize=8)
    ax.set_ylim(0, max(sr['means'])*1.35)
    ax.tick_params(axis='x', labelsize=7)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.grid(axis='y', color='#e8e8e8', linewidth=0.6)

fig.tight_layout()
fig.savefig('osm_F_sensitivity.png', bbox_inches='tight', dpi=300)
plt.close()
print("  Saved: osm_F_sensitivity.png")

# Calibration summary chart
fig, ax = plt.subplots(figsize=(5, 3))
ax.axhline(y=TARGET_BASELINE, color=PS_DARK, linewidth=1.5,
           linestyle='--', label=f'Target: {TARGET_BASELINE} min (Shoup 2006)')
ax.scatter([S2M_CALIBRATED], [run_baseline(S2M_CALIBRATED, 10)],
           color=PS_MID, s=80, zorder=5, label='Calibrated S2M')
ax.set_xlabel('S2M (min/step)')
ax.set_ylabel('Baseline search time (min)')
ax.set_title('Auto-calibration convergence\n'
             f'Final S2M = {S2M_CALIBRATED:.4f}', loc='center')
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig('osm_G_calibration.png', bbox_inches='tight', dpi=300)
plt.close()
print("  Saved: osm_G_calibration.png")

print(f"""
════════════════════════════════════════════════════════════
  OSM SIMULATION COMPLETE

  Calibrated S2M = {S2M_CALIBRATED:.4f} min/step
  Baseline on OSM network ≈ {run_baseline(S2M_CALIBRATED, 10):.2f} min (target: 8.6)

  Charts:
  osm_A_search_time.png  → Search time with 95% CI
  osm_B_distance.png     → Real street distances
  osm_C_cost.png         → Cost per driver
  osm_D_co2.png          → CO₂ (real distances)
  osm_E_significance.png → t-test results
  osm_F_sensitivity.png  → Sensitivity analysis
  osm_G_calibration.png  → Calibration convergence

  Data:
  parkshare_osm_full_results.csv
  parkshare_osm_stats.csv
════════════════════════════════════════════════════════════
""")
